In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetB0
import warnings
warnings.filterwarnings('ignore')
 
print("✓ Python version     :", sys.version[:6])
print("✓ TensorFlow version :", tf.__version__)
print("✓ GPU détecté        :", tf.config.list_physical_devices('GPU'))
print("✓ Imports réussis")
 
# ── Chemins du projet ─────────────────────────────────────
BASE_DIR   = Path(r"C:\Users\ramatoulaye sy\skin_cancer_project")
DATA_DIR   = BASE_DIR / "data" / "melanoma_cancer_dataset"
TRAIN_DIR  = DATA_DIR / "train"
TEST_DIR   = DATA_DIR / "test"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
 
# ── Paramètres globaux ────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 16
SEED       = 42
EPOCHS     = 20
 
print(f"\n✓ Dossier modèles    : {MODELS_DIR}")
print(f"✓ IMG_SIZE           : {IMG_SIZE}")
print(f"✓ BATCH_SIZE         : {BATCH_SIZE}")
print(f"✓ EPOCHS             : {EPOCHS}")
 

✓ Python version     : 3.10.2
✓ TensorFlow version : 2.21.0
✓ GPU détecté        : []
✓ Imports réussis

✓ Dossier modèles    : C:\Users\ramatoulaye sy\skin_cancer_project\models
✓ IMG_SIZE           : 224
✓ BATCH_SIZE         : 16
✓ EPOCHS             : 20


In [2]:
TRAIN_BENIGN    = TRAIN_DIR / "benign"
TRAIN_MALIGNANT = TRAIN_DIR / "malignant"
TEST_BENIGN     = TEST_DIR  / "benign"
TEST_MALIGNANT  = TEST_DIR  / "malignant"
 
def construire_dataframe(dossier_benin, dossier_malin):
    lignes = []
    for img_path in dossier_benin.glob("*.*"):
        lignes.append({'image_path': str(img_path), 'label': '0', 'target': 0})
    for img_path in dossier_malin.glob("*.*"):
        lignes.append({'image_path': str(img_path), 'label': '1', 'target': 1})
    return pd.DataFrame(lignes).sample(frac=1, random_state=SEED).reset_index(drop=True)
 
# Chargement depuis CSV si disponible, sinon recréation
csv_train = BASE_DIR / "data" / "df_train.csv"
csv_val   = BASE_DIR / "data" / "df_val.csv"
csv_test  = BASE_DIR / "data" / "df_test.csv"
 
if csv_train.exists() and csv_val.exists() and csv_test.exists():
    df_train = pd.read_csv(csv_train)
    df_val   = pd.read_csv(csv_val)
    df_test  = pd.read_csv(csv_test)
    print("✓ Splits chargés depuis les CSV sauvegardés")
else:
    df_trainval = construire_dataframe(TRAIN_BENIGN, TRAIN_MALIGNANT)
    df_test     = construire_dataframe(TEST_BENIGN,  TEST_MALIGNANT)
    df_train, df_val = train_test_split(
        df_trainval, test_size=0.15, random_state=SEED,
        stratify=df_trainval['target']
    )
    print("✓ Splits recréés depuis les dossiers")
 
# Assure que label est bien string
df_train['label'] = df_train['label'].astype(str)
df_val['label']   = df_val['label'].astype(str)
df_test['label']  = df_test['label'].astype(str)
 
# Class weights
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=df_train['target'].values
)
CLASS_WEIGHT_DICT = {0: class_weights_array[0], 1: class_weights_array[1]}
 
print(f"\n  Train      : {len(df_train)} images")
print(f"  Validation : {len(df_val)} images")
print(f"  Test       : {len(df_test)} images")
print(f"\n  Class weights : {CLASS_WEIGHT_DICT}")
 

✓ Splits chargés depuis les CSV sauvegardés

  Train      : 8164 images
  Validation : 1441 images
  Test       : 1000 images

  Class weights : {0: np.float64(0.9604705882352941), 1: np.float64(1.0429228410832907)}


In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)
 
val_test_datagen = ImageDataGenerator(rescale=1./255)
 
train_generator = train_datagen.flow_from_dataframe(
    dataframe=df_train, x_col='image_path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True, seed=SEED
)
 
val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_val, x_col='image_path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)
 
test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_test, x_col='image_path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)
 
print("✓ Générateurs prêts")
print(f"  Train      : {train_generator.samples} images")
print(f"  Validation : {val_generator.samples} images")
print(f"  Test       : {test_generator.samples} images")
 
 

Found 8164 validated image filenames belonging to 2 classes.
Found 1441 validated image filenames belonging to 2 classes.
Found 1000 validated image filenames belonging to 2 classes.
✓ Générateurs prêts
  Train      : 8164 images
  Validation : 1441 images
  Test       : 1000 images


In [4]:
def creer_cnn_base(img_size=224):
    """Crée un CNN simple from scratch."""
    model = models.Sequential([
 
        # ── Bloc 1 : extraction de features basiques ──────
        layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                      input_shape=(img_size, img_size, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
 
        # ── Bloc 2 : features intermédiaires ──────────────
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
 
        # ── Bloc 3 : features complexes ───────────────────
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
 
        # ── Bloc 4 : features très complexes ──────────────
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
 
        # ── Classification ────────────────────────────────
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),             # évite le surapprentissage
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')  # sortie binaire [0,1]
    ], name='CNN_Base')
 
    return model
 
# Création et compilation
cnn_base = creer_cnn_base(IMG_SIZE)
 
cnn_base.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)
 
# Résumé du modèle
cnn_base.summary()
print(f"\n✓ CNN de base créé")
print(f"  Paramètres totaux : {cnn_base.count_params():,}")
 
 

Model: "CNN_Base"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 224, 224, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 224, 224, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 112, 112, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 112, 112, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 112, 112, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 56, 56, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 56, 56, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 56, 56, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 28, 28, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 28, 28, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 28, 28, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 14, 14, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 256)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │          65,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 490,177 (1.87 MB)

 Trainable params: 488,705 (1.86 MB)

 Non-trainable params: 1,472 (5.75 KB)


✓ CNN de base créé
  Paramètres totaux : 490,177


In [5]:
def creer_callbacks(nom_modele):
    """Crée les callbacks pour un modèle donné."""
 
    # 1. Sauvegarde du meilleur modèle (selon val_auc)
    checkpoint = ModelCheckpoint(
        filepath=str(MODELS_DIR / f"{nom_modele}_best.keras"),
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    )
 
    # 2. Arrêt anticipé si pas d'amélioration pendant 5 epochs
    early_stop = EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
 
    # 3. Réduction du learning rate si plateau
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,          # divise le lr par 2
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
 
    return [checkpoint, early_stop, reduce_lr]
 
callbacks_base = creer_callbacks("cnn_base")
print("✓ Callbacks créés :")
print("  - ModelCheckpoint  → sauvegarde le meilleur modèle")
print("  - EarlyStopping    → arrêt si pas d'amélioration (patience=5)")
print("  - ReduceLROnPlateau→ réduit le learning rate si plateau (patience=3)")
 
 

✓ Callbacks créés :
  - ModelCheckpoint  → sauvegarde le meilleur modèle
  - EarlyStopping    → arrêt si pas d'amélioration (patience=5)
  - ReduceLROnPlateau→ réduit le learning rate si plateau (patience=3)


In [6]:
print("=" * 52)
print("  DÉMARRAGE DE L'ENTRAÎNEMENT — CNN DE BASE")
print("=" * 52)
print(f"  Epochs max   : {EPOCHS}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Class weights: {CLASS_WEIGHT_DICT}")
print("=" * 52)
 
history_base = cnn_base.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    class_weight=CLASS_WEIGHT_DICT,
    callbacks=callbacks_base,
    verbose=1
)
 
print("\n✓ Entraînement terminé !")
print(f"  Modèle sauvegardé : {MODELS_DIR}/cnn_base_best.keras")
 

  DÉMARRAGE DE L'ENTRAÎNEMENT — CNN DE BASE
  Epochs max   : 20
  Batch size   : 16
  Class weights: {0: np.float64(0.9604705882352941), 1: np.float64(1.0429228410832907)}
Epoch 1/20
511/511 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7483 - auc: 0.8149 - loss: 0.6006 - precision: 0.7408 - recall: 0.7383
Epoch 1: val_auc improved from None to 0.91898, saving model to C:\Users\ramatoulaye sy\skin_cancer_project\models\cnn_base_best.keras
511/511 ━━━━━━━━━━━━━━━━━━━━ 1023s 2s/step - accuracy: 0.7705 - auc: 0.8458 - loss: 0.5277 - precision: 0.7615 - recall: 0.7588 - val_accuracy: 0.8203 - val_auc: 0.9190 - val_loss: 0.3955 - val_precision: 0.9235 - val_recall: 0.6816 - learning_rate: 0.0010
Epoch 2/20
511/511 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8196 - auc: 0.8957 - loss: 0.4184 - precision: 0.8127 - recall: 0.8109
Epoch 2: val_auc improved from 0.91898 to 0.93439, saving model to C:\Users\ramatoulaye sy\skin_cancer_project\models\cnn_base_best.keras
511/511 ━━━━━━━━━━━━━━━━━━

In [10]:
%matplotlib inline
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

history = history_base

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Courbes d'apprentissage — CNN de base", fontsize=14, fontweight='bold')

axes[0].plot(history.history['loss'],     label='Train',      color='#1A8A56', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation', color='#E24B4A', linewidth=2)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train',      color='#1A8A56', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation', color='#E24B4A', linewidth=2)
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history.history['auc'],     label='Train',      color='#1A8A56', linewidth=2)
axes[2].plot(history.history['val_auc'], label='Validation', color='#E24B4A', linewidth=2)
axes[2].set_title('AUC-ROC')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

# Sauvegarde sur le Bureau
save_path = r"C:\Users\ramatoulaye sy\Desktop\courbes_CNN_Base.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close()

# Vérification
if os.path.exists(save_path):
    print(f"✓ Fichier créé sur le Bureau !")
    print(f"  Chemin : {save_path}")
else:
    print("✗ Échec — affichage des clés disponibles :")
    print(list(history.history.keys()))

✓ Fichier créé sur le Bureau !
  Chemin : C:\Users\ramatoulaye sy\Desktop\courbes_CNN_Base.png


In [11]:
def creer_efficientnet(img_size=224):
    """Crée un modèle EfficientNetB0 avec Transfer Learning."""
 
    # Chargement du modèle pré-entraîné (sans la tête de classification)
    base_model = EfficientNetB0(
        weights='imagenet',         # poids pré-entraînés sur ImageNet
        include_top=False,          # on retire la couche de sortie originale
        input_shape=(img_size, img_size, 3)
    )
 
    # Gel de toutes les couches de base (phase 1 : feature extraction)
    base_model.trainable = False
 
    print(f"  Couches de EfficientNetB0 : {len(base_model.layers)}")
    print(f"  Couches gelées            : {len(base_model.layers)}")
 
    # Construction du modèle complet
    inputs = keras.Input(shape=(img_size, img_size, 3))
 
    # Passage dans EfficientNetB0 (gelé)
    x = base_model(inputs, training=False)
 
    # Nos couches de classification personnalisées
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
 
    model = keras.Model(inputs, outputs, name='EfficientNetB0_TL')
    return model, base_model
 
# Création du modèle
efficientnet, base_model = creer_efficientnet(IMG_SIZE)
 
efficientnet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)
 
efficientnet.summary()
print(f"\n✓ EfficientNetB0 créé")
print(f"  Paramètres totaux    : {efficientnet.count_params():,}")
print(f"  Paramètres entraînables : {sum([tf.size(w).numpy() for w in efficientnet.trainable_weights]):,}")
 
 

  Couches de EfficientNetB0 : 238
  Couches gelées            : 238


Model: "EfficientNetB0_TL"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ efficientnetb0 (Functional)          │ (None, 7, 7, 1280)          │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_2           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_6                │ (None, 1280)                │           5,120 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,415,652 (16.84 MB)

 Trainable params: 363,521 (1.39 MB)

 Non-trainable params: 4,052,131 (15.46 MB)


✓ EfficientNetB0 créé
  Paramètres totaux    : 4,415,652
  Paramètres entraînables : 363,521


In [2]:
print("=" * 52)
print("  PHASE 1 — ENTRAÎNEMENT DE LA TÊTE SEULEMENT")
print("=" * 52)
 
callbacks_eff_phase1 = creer_callbacks("efficientnet_phase1")
 
history_eff_phase1 = efficientnet.fit(
    train_generator,
    epochs=10,                      # 10 epochs pour la phase 1
    validation_data=val_generator,
    class_weight=CLASS_WEIGHT_DICT,
    callbacks=callbacks_eff_phase1,
    verbose=1
)
 
print("\n✓ Phase 1 terminée !")
#afficher_courbes(history_eff_phase1, "EfficientNet_Phase1")
 
 

  PHASE 1 — ENTRAÎNEMENT DE LA TÊTE SEULEMENT


NameError: name 'creer_callbacks' is not defined

In [1]:
print("=" * 52)
print("  PHASE 1 — ENTRAÎNEMENT DE LA TÊTE SEULEMENT")
print("=" * 52)
 
callbacks_eff_phase1 = creer_callbacks("efficientnet_phase1")
 
history_eff_phase1 = efficientnet.fit(
    train_generator,
    epochs=10,                      # 10 epochs pour la phase 1
    validation_data=val_generator,
    class_weight=CLASS_WEIGHT_DICT,
    callbacks=callbacks_eff_phase1,
    verbose=1
)
 
print("\n✓ Phase 1 terminée !")
afficher_courbes(history_eff_phase1, "EfficientNet_Phase1")
# CORRECTION — Regénérer les générateurs pour EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# EfficientNetB0 utilise son propre prétraitement (pas rescale=1./255)
train_datagen_eff = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)
val_test_datagen_eff = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen_eff.flow_from_dataframe(
    dataframe=df_train, x_col='image_path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True, seed=SEED
)
val_generator = val_test_datagen_eff.flow_from_dataframe(
    dataframe=df_val, x_col='image_path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)
test_generator = val_test_datagen_eff.flow_from_dataframe(
    dataframe=df_test, x_col='image_path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)
print("✓ Générateurs corrigés pour EfficientNetB0 !")
 
 

  PHASE 1 — ENTRAÎNEMENT DE LA TÊTE SEULEMENT


NameError: name 'creer_callbacks' is not defined

In [ ]:
def evaluer_modele(model, generator, nom_modele):
    """Évalue un modèle et affiche toutes les métriques."""
    print(f"\n{'='*52}")
    print(f"  ÉVALUATION — {nom_modele}")
    print(f"{'='*52}")
 
    # Prédictions
    generator.reset()
    y_pred_proba = model.predict(generator, verbose=1)
    y_pred       = (y_pred_proba > 0.5).astype(int).flatten()
    y_true       = generator.labels.astype(int)
 
    # Métriques
    auc    = roc_auc_score(y_true, y_pred_proba)
    report = classification_report(y_true, y_pred,
                                   target_names=['Bénin', 'Malin'])
 
    print(f"\n  AUC-ROC : {auc:.4f}")
    print(f"\n  Rapport de classification :\n")
    print(report)
 
    # Matrice de confusion
    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Évaluation — {nom_modele}', fontsize=14, fontweight='bold')
 
    # Heatmap matrice de confusion
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Bénin', 'Malin'],
                yticklabels=['Bénin', 'Malin'],
                ax=axes[0])
    axes[0].set_title('Matrice de confusion')
    axes[0].set_ylabel('Valeur réelle')
    axes[0].set_xlabel('Valeur prédite')
 
    # Courbe ROC
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    axes[1].plot(fpr, tpr, color='#3B8BD4', linewidth=2,
                 label=f'AUC = {auc:.4f}')
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aléatoire')
    axes[1].set_title('Courbe ROC')
    axes[1].set_xlabel('Taux de faux positifs')
    axes[1].set_ylabel('Taux de vrais positifs (Sensibilité)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
 
    plt.tight_layout()
    save_path = BASE_DIR / "notebooks" / f"evaluation_{nom_modele}.png"
    plt.savefig(str(save_path), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Graphique sauvegardé : {save_path}")
 
    return auc, y_pred, y_true
 
# Évaluation des deux modèles
auc_base, _, _ = evaluer_modele(cnn_base,      test_generator, "CNN_Base")
auc_eff,  _, _ = evaluer_modele(efficientnet,  test_generator, "EfficientNetB0")

In [15]:
print("=" * 52)
print("  COMPARAISON DES MODÈLES")
print("=" * 52)
print(f"\n  {'Modèle':<30} {'AUC-ROC':>10}")
print(f"  {'─'*42}")
print(f"  {'CNN de base':<30} {auc_base:>10.4f}")
print(f"  {'EfficientNetB0 (Transfer Learning)':<30} {auc_eff:>10.4f}")
 
meilleur = "EfficientNetB0" if auc_eff > auc_base else "CNN de base"
print(f"\n  ✓ Meilleur modèle : {meilleur}")
 
# Sauvegarde finale du meilleur modèle
if auc_eff > auc_base:
    efficientnet.save(str(MODELS_DIR / "meilleur_modele.keras"))
    print(f"  ✓ EfficientNetB0 sauvegardé comme meilleur modèle")
else:
    cnn_base.save(str(MODELS_DIR / "meilleur_modele.keras"))
    print(f"  ✓ CNN de base sauvegardé comme meilleur modèle")
 
print(f"\n  Fichier : {MODELS_DIR}/meilleur_modele.keras")
print("\n" + "=" * 52)
print("  ✓ PHASE 3 TERMINÉE AVEC SUCCÈS !")
print("  → Prochaine étape : Phase 4 — Application web")
print("=" * 52)
 

  COMPARAISON DES MODÈLES

  Modèle                            AUC-ROC
  ──────────────────────────────────────────


NameError: name 'auc_base' is not defined

In [16]:
# DIAGNOSTIC
print("Clés disponibles :", list(history_base.history.keys()))
print("Nombre d'epochs  :", len(history_base.history['loss']))
print("Valeurs loss     :", history_base.history['loss'])

Clés disponibles : ['accuracy', 'auc', 'loss', 'precision', 'recall', 'val_accuracy', 'val_auc', 'val_loss', 'val_precision', 'val_recall', 'learning_rate']
Nombre d'epochs  : 20
Valeurs loss     : [0.5276619791984558, 0.41591912508010864, 0.37568822503089905, 0.359000563621521, 0.3355364203453064, 0.312443345785141, 0.2972775101661682, 0.29624783992767334, 0.29237985610961914, 0.2808215022087097, 0.28333780169487, 0.28360262513160706, 0.27910956740379333, 0.26785776019096375, 0.25820696353912354, 0.2594910264015198, 0.253985732793808, 0.2529507875442505, 0.2486056238412857, 0.24579358100891113]


In [4]:
# ÉVALUATION FINALE — CNN de base
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path

MODELS_DIR = Path(r"C:\Users\ramatoulaye sy\skin_cancer_project\models")

# Chargement du meilleur modèle
cnn_base = tf.keras.models.load_model(str(MODELS_DIR / "cnn_base_best.keras"))
print("✓ Modèle chargé !")

# Prédictions sur le Test
test_generator.reset()
y_pred_proba = cnn_base.predict(test_generator, verbose=1)
y_pred       = (y_pred_proba > 0.5).astype(int).flatten()
y_true       = test_generator.labels.astype(int)

# Métriques
auc    = roc_auc_score(y_true, y_pred_proba)
report = classification_report(y_true, y_pred, target_names=['Bénin','Malin'])

print(f"\n{'='*50}")
print(f"  AUC-ROC  : {auc:.4f}")
print(f"\n{report}")
print(f"{'='*50}")

# Matrice de confusion + Courbe ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Bénin','Malin'],
            yticklabels=['Bénin','Malin'], ax=axes[0])
axes[0].set_title('Matrice de confusion')
axes[0].set_ylabel('Réel')
axes[0].set_xlabel('Prédit')

fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
axes[1].plot(fpr, tpr, color='#1A8A56', linewidth=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_title('Courbe ROC')
axes[1].set_xlabel('Faux positifs')
axes[1].set_ylabel('Vrais positifs')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r"C:\Users\ramatoulaye sy\Desktop\evaluation_finale.png", dpi=150)
plt.close()
print("\n✓ Graphique sauvegardé sur le Bureau !")

# Sauvegarde modèle final
cnn_base.save(str(MODELS_DIR / "meilleur_modele.keras"))
print("✓ Modèle final sauvegardé : meilleur_modele.keras")

✓ Modèle chargé !


NameError: name 'test_generator' is not defined

In [6]:
# ÉVALUATION FINALE — CNN de base
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ── Chemins ──────────────────────────────────────────────
BASE_DIR   = Path(r"C:\Users\ramatoulaye sy\skin_cancer_project")
MODELS_DIR = BASE_DIR / "models"
DATA_DIR   = BASE_DIR / "data"

# ── Rechargement des données ──────────────────────────────
df_test = pd.read_csv(str(DATA_DIR / "df_test.csv"))
df_test['label'] = df_test['target'].astype(str)
print(f"✓ df_test chargé : {len(df_test)} images")

# ── Générateur test ───────────────────────────────────────
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=df_test,
    x_col='image_path',
    y_col='label',
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary',
    shuffle=False
)
print("✓ Générateur test créé !")

# ── Chargement du modèle ──────────────────────────────────
cnn_base = tf.keras.models.load_model(str(MODELS_DIR / "cnn_base_best.keras"))
print("✓ Modèle chargé !")

# ── Prédictions ───────────────────────────────────────────
test_generator.reset()
y_pred_proba = cnn_base.predict(test_generator, verbose=1)
y_pred       = (y_pred_proba > 0.5).astype(int).flatten()
y_true = np.array(test_generator.labels).astype(int)

# ── Métriques ─────────────────────────────────────────────
auc    = roc_auc_score(y_true, y_pred_proba)
report = classification_report(y_true, y_pred, target_names=['Bénin','Malin'])

print(f"\n{'='*50}")
print(f"  AUC-ROC  : {auc:.4f}")
print(f"\n{report}")
print(f"{'='*50}")

# ── Graphiques ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Bénin','Malin'],
            yticklabels=['Bénin','Malin'], ax=axes[0])
axes[0].set_title('Matrice de confusion')
axes[0].set_ylabel('Réel')
axes[0].set_xlabel('Prédit')

fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
axes[1].plot(fpr, tpr, color='#1A8A56', linewidth=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_title('Courbe ROC')
axes[1].set_xlabel('Faux positifs')
axes[1].set_ylabel('Vrais positifs')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r"C:\Users\ramatoulaye sy\Desktop\evaluation_finale.png", dpi=150)
plt.close()
print("\n✓ Graphique sauvegardé sur le Bureau !")

# ── Sauvegarde modèle final ───────────────────────────────
cnn_base.save(str(MODELS_DIR / "meilleur_modele.keras"))
print("✓ Modèle final sauvegardé : meilleur_modele.keras")

✓ df_test chargé : 1000 images
Found 1000 validated image filenames belonging to 2 classes.
✓ Générateur test créé !
✓ Modèle chargé !
63/63 ━━━━━━━━━━━━━━━━━━━━ 19s 297ms/step

  AUC-ROC  : 0.9674

              precision    recall  f1-score   support

       Bénin       0.86      0.97      0.91       500
       Malin       0.97      0.84      0.90       500

    accuracy                           0.91      1000
   macro avg       0.91      0.91      0.91      1000
weighted avg       0.91      0.91      0.91      1000


✓ Graphique sauvegardé sur le Bureau !
✓ Modèle final sauvegardé : meilleur_modele.keras


In [3]:
# CORRECTION GRAD-CAM — Vérification des couches Conv2D
import tensorflow as tf
from pathlib import Path

model = tf.keras.models.load_model(
    str(Path(r"C:\Users\ramatoulaye sy\skin_cancer_project\models\meilleur_modele.keras"))
)

# Affiche toutes les couches du modèle
print("=== Couches du modèle ===")
for i, layer in enumerate(model.layers):
    print(f"  {i:>3} | {layer.name:<40} | {type(layer).__name__}")

=== Couches du modèle ===
    0 | conv2d                                   | Conv2D
    1 | batch_normalization                      | BatchNormalization
    2 | max_pooling2d                            | MaxPooling2D
    3 | conv2d_1                                 | Conv2D
    4 | batch_normalization_1                    | BatchNormalization
    5 | max_pooling2d_1                          | MaxPooling2D
    6 | conv2d_2                                 | Conv2D
    7 | batch_normalization_2                    | BatchNormalization
    8 | max_pooling2d_2                          | MaxPooling2D
    9 | conv2d_3                                 | Conv2D
   10 | batch_normalization_3                    | BatchNormalization
   11 | max_pooling2d_3                          | MaxPooling2D
   12 | global_average_pooling2d                 | GlobalAveragePooling2D
   13 | dense                                    | Dense
   14 | batch_normalization_4                    | BatchNormalization
   15

In [2]:
# Vérification des couches du modèle chargé
import tensorflow as tf
from pathlib import Path

model = tf.keras.models.load_model(
    str(Path(r"C:\Users\ramatoulaye sy\skin_cancer_project\models\meilleur_modele.keras"))
)

print("=== Couches Conv2D disponibles ===")
for i, layer in enumerate(model.layers):
    if 'conv' in layer.name.lower():
        print(f"  {i:>3} | {layer.name:<40} | {layer.output.shape}")

=== Couches Conv2D disponibles ===
    0 | conv2d                                   | (None, 224, 224, 32)
    3 | conv2d_1                                 | (None, 112, 112, 64)
    6 | conv2d_2                                 | (None, 56, 56, 128)
    9 | conv2d_3                                 | (None, 28, 28, 256)


In [4]:
# Diagnostic complet
print("=== Couches du modèle principal ===")
for i, layer in enumerate(model.layers):
    print(f"{i:>3} | {layer.name} | {type(layer).__name__}")

# Chercher le sous-modèle
for layer in model.layers:
    if hasattr(layer, 'layers'):  # C'est un sous-modèle
        print(f"\n=== Couches dans '{layer.name}' ===")
        for j, sublayer in enumerate(layer.layers):
            print(f"{j:>3} | {sublayer.name} | {type(sublayer).__name__}")

=== Couches du modèle principal ===
  0 | conv2d | Conv2D
  1 | batch_normalization | BatchNormalization
  2 | max_pooling2d | MaxPooling2D
  3 | conv2d_1 | Conv2D
  4 | batch_normalization_1 | BatchNormalization
  5 | max_pooling2d_1 | MaxPooling2D
  6 | conv2d_2 | Conv2D
  7 | batch_normalization_2 | BatchNormalization
  8 | max_pooling2d_2 | MaxPooling2D
  9 | conv2d_3 | Conv2D
 10 | batch_normalization_3 | BatchNormalization
 11 | max_pooling2d_3 | MaxPooling2D
 12 | global_average_pooling2d | GlobalAveragePooling2D
 13 | dense | Dense
 14 | batch_normalization_4 | BatchNormalization
 15 | dropout | Dropout
 16 | dense_1 | Dense
 17 | dropout_1 | Dropout
 18 | dense_2 | Dense


In [6]:
import tensorflow as tf
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Charger le modèle
model = tf.keras.models.load_model(
    str(Path(r"C:\Users\ramatoulaye sy\skin_cancer_project\models\meilleur_modele.keras")),
    compile=False
)

# Forcer la construction
dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
_ = model(dummy, training=False)

# Charger l'image
img_path = r"C:\Users\ramatoulaye sy\skin_cancer_project\data\melanoma_cancer_dataset\train\malignant\melanoma_9591.jpg"
image = Image.open(img_path).convert('RGB')
img_r = np.array(image.resize((224, 224)))
img_array = tf.cast(np.expand_dims(img_r / 255.0, axis=0), tf.float32)

try:
    LAST_CONV_LAYER = "conv2d_3"
    last_conv = model.get_layer(LAST_CONV_LAYER)

    # Créer un modèle intermédiaire couche par couche
    # sans passer par model.inputs
    @tf.function
    def get_grad_cam(img):
        with tf.GradientTape() as tape:
            x = img
            conv_out = None
            for layer in model.layers:
                x = layer(x, training=False)
                if layer.name == LAST_CONV_LAYER:
                    conv_out = x
                    tape.watch(conv_out)
            loss = x[:, 0]
        grads = tape.gradient(loss, conv_out)
        return conv_out, grads, x

    conv_outputs, grads, predictions = get_grad_cam(img_array)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= (tf.reduce_max(heatmap) + 1e-8)
    heatmap = heatmap.numpy()

    # Superposition
    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    superimposed = cv2.addWeighted(img_r, 0.6, heatmap, 0.4, 0)

    # Sauvegarde
    save_path = r"C:\Users\ramatoulaye sy\Desktop\test_gradcam.png"
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img_r)
    plt.title("Image originale")
    plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(superimposed)
    plt.title("Grad-CAM")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

    print("✓ Grad-CAM généré avec succès !")
    print(f"✓ Sauvegardé : {save_path}")
    print(f"✓ Prédiction : {float(predictions[0][0]):.4f}")

except Exception as e:
    import traceback
    print(f"✗ Erreur : {e}")
    traceback.print_exc()

✓ Grad-CAM généré avec succès !
✓ Sauvegardé : C:\Users\ramatoulaye sy\Desktop\test_gradcam.png
✓ Prédiction : 0.9525
